In [ ]:
# ===============================
# 1️⃣ استيراد المكتبات
# ===============================
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# ===============================
# 2️⃣ تحميل البيانات
# ===============================
file_path = r"C:\Users\INFOTEC\OneDrive\Bureau\ML_ready_predition.csv"
df = pd.read_csv(file_path)

# ===============================
# 3️⃣ تقسيم train/test حسب الأسبوع
# ===============================
# train = الأسابيع 1،2،3
# test = الأسبوع 4
train_weeks = [1,2,3]
test_weeks = [4]

df_train = df[df['Week'].isin(train_weeks)]
df_test = df[df['Week'].isin(test_weeks)]

# ===============================
# 4️⃣ إعداد Features و Target
# ===============================
features = [
    'Day_Index', 'Week', 'Week_of_Year', 'Country', 'Part_Number',
    'Stock_Before', 'Usage_MA3', 'Daily_Usage_Mean', 'Usage_Trend',
    'CV_Inter', 'Safety_Stock', 'Reorder_Point', 'ABC_Class', 'XYZ_Class', 
    'Rotation_Rate', 'Coverage_Days'
]

# تحويل التصنيفات لرقمي
X_train = pd.get_dummies(df_train[features], columns=['ABC_Class','XYZ_Class','Country','Part_Number'])
y_train = df_train['Daily_Usage']

X_test = pd.get_dummies(df_test[features], columns=['ABC_Class','XYZ_Class','Country','Part_Number'])
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# ===============================
# 5️⃣ تدريب الموديل
# ===============================
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# ===============================
# 6️⃣ التنبؤ على الأسبوع الرابع
# ===============================
predicted_usage = model.predict(X_test)
df_test['Predicted_Usage'] = predicted_usage

# تقييم الأداء
mse = mean_squared_error(df_test['Daily_Usage'], predicted_usage)
print("Mean Squared Error:", mse)

# ===============================
# 7️⃣ حساب Stock_Future و Alerts و Quantité à Commander
# ===============================
Lead_Time = 3  # يمكن تغييره حسب مشروعك
stock_future_list = []
alert_list = []
quantity_list = []

for i, row in df_test.iterrows():
    stock_today = row['Stock_Before']
    predicted = row['Predicted_Usage']
    reorder_point = row['Reorder_Point']
    safety_stock = row['Safety_Stock']
    
    stock_future = stock_today - predicted
    stock_future_list.append(stock_future)
    
    alert = stock_future <= reorder_point
    alert_list.append(alert)
    
    quantity = max(0, (reorder_point - stock_future) + safety_stock)
    quantity_list.append(quantity)

df_test['Stock_Future'] = stock_future_list
df_test['Reorder_Alert'] = alert_list
df_test['Quantité_Commander'] = quantity_list

# ===============================
# 8️⃣ عرض النتائج
# ===============================
print(df_test[['Date','Day_Index','Week','Stock_Before','Predicted_Usage','Stock_Future','Reorder_Alert','Quantité_Commander']])

# ===============================
# 9️⃣ حفظ النتائج في CSV جديد
# ===============================
output_file = r"C:\Users\INFOTEC\OneDrive\Bureau\week4_predicted_results.csv"
df_test.to_csv(output_file, index=False)
print(f"Results saved to {output_file}")